In [65]:
## install finrl library
!pip install wrds
!pip install swig
!pip install -q condacolab
import condacolab
condacolab.install()
!apt-get update -y -qq && apt-get install -y -qq cmake libopenmpi-dev python3-dev zlib1g-dev libgl1-mesa-glx swig
!pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

/usr/local/lib/python3.11/site-packages/condacolab.py:320: DeprecationWarning: Use shutil.which instead of find_executable
  assert find_executable("conda"), "💥💔💥 Conda not found!"


✨🍰✨ Everything looks OK!
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
  Cloning https://github.com/AI4Finance-Foundation/FinRL.git to /tmp/pip-req-build-214aywar
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/FinRL.git /tmp/pip-req-build-214aywar
  Resolved https://github.com/AI4Finance-Foundation/FinRL.git to commit 3a10805389c07e13e94333e94dcfe9beafce49b0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Cloning https://github.com/AI4Finance-Foundation/ElegantRL.git to /tmp/pip-install-6swh12gd/elegantrl_b047a58a4cb242d98be28bca0be79940
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/ElegantRL.git /tmp/pip-install-6swh12gd/elegantrl_b047a58a4cb242d98be28bca0be79

In [66]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.use('Agg')
%matplotlib inline
import datetime

from finrl import config
from finrl import config_tickers
from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.meta.env_portfolio_allocation.env_portfolio import StockPortfolioEnv
from finrl.agents.stablebaselines3.models import DRLAgent
from finrl.plot import backtest_stats, backtest_plot, get_daily_return, get_baseline,convert_daily_return_to_pyfolio_ts
from finrl.meta.data_processor import DataProcessor
from finrl.meta.data_processors.processor_yahoofinance import YahooFinanceProcessor
import sys
sys.path.append("../FinRL-Library")

In [67]:
import os
if not os.path.exists("./" + config.DATA_SAVE_DIR):
    os.makedirs("./" + config.DATA_SAVE_DIR)
if not os.path.exists("./" + config.TRAINED_MODEL_DIR):
    os.makedirs("./" + config.TRAINED_MODEL_DIR)
if not os.path.exists("./" + config.TENSORBOARD_LOG_DIR):
    os.makedirs("./" + config.TENSORBOARD_LOG_DIR)
if not os.path.exists("./" + config.RESULTS_DIR):
    os.makedirs("./" + config.RESULTS_DIR)

In [68]:
print(config_tickers.DOW_30_TICKER)

['AXP', 'AMGN', 'AAPL', 'BA', 'CAT', 'CSCO', 'CVX', 'GS', 'HD', 'HON', 'IBM', 'INTC', 'JNJ', 'KO', 'JPM', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'PG', 'TRV', 'UNH', 'CRM', 'VZ', 'V', 'WBA', 'WMT', 'DIS', 'DOW']


In [69]:
df=pd.read_csv('/content/TSX Chart Data Sorted By Date.csv')

In [70]:
df.head()

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,date,close,high,low,open,volume,tic,day
0,2004-01-01,0.105000,0.105,0.105,0.105,0.0,DYA.TO,3.0
1,2004-01-01,1.378000,2.650,2.650,2.650,0.0,EDT.TO,3.0
2,2004-01-02,0.489625,0.530,0.530,0.530,0.0,AAB.TO,4.0
3,2004-01-02,22.070272,29.850,29.320,29.430,413000.0,ABX.TO,4.0
4,2004-01-02,3.484277,7.050,7.050,7.050,100.0,ACD.TO,4.0


In [71]:
df.shape

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


(117649, 8)

In [72]:
print(df.columns)

Index(['date', 'close', 'high', 'low', 'open', 'volume', 'tic', 'day'], dtype='object')


In [73]:
df.shape

(117649, 8)

In [74]:
df.head()

,date,close,high,low,open,volume,tic,day
0,2004-01-01,0.105000,0.105,0.105,0.105,0.0,DYA.TO,3.0
1,2004-01-01,1.378000,2.650,2.650,2.650,0.0,EDT.TO,3.0
2,2004-01-02,0.489625,0.530,0.530,0.530,0.0,AAB.TO,4.0
3,2004-01-02,22.070272,29.850,29.320,29.430,413000.0,ABX.TO,4.0
4,2004-01-02,3.484277,7.050,7.050,7.050,100.0,ACD.TO,4.0


In [75]:
# add covariance matrix as states
df=df.sort_values(['date','tic'],ignore_index=True)
df.index = df.date.factorize()[0]

cov_list = []
return_list = []

# look back is one year
lookback=252
for i in range(lookback,len(df.index.unique())):
  data_lookback = df.loc[i-lookback:i,:]
  price_lookback=data_lookback.pivot_table(index = 'date',columns = 'tic', values = 'close')
  return_lookback = price_lookback.pct_change().dropna()
  return_list.append(return_lookback)

  covs = return_lookback.cov().values
  cov_list.append(covs)


df_cov = pd.DataFrame({'date':df.date.unique()[lookback:],'cov_list':cov_list,'return_list':return_list})
df = df.merge(df_cov, on='date')
df = df.sort_values(['date','tic']).reset_index(drop=True)

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
<ipython-input-75-7c9fed0ab95b>:13: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return_lookback = price_lookback.pct_change().dropna()
<ipython-input-75-7c9fed0ab95b>:13: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  return_lookback = pric

In [76]:
df.shape

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


(95153, 10)

In [77]:
df.head()

,date,close,high,low,open,volume,tic,day,cov_list,return_list
0,2004-12-20,0.415720,0.450000,0.450000,0.450000,0.0,AAB.TO,0.0,"[[0.009335785113573457, -0.0001770641735861586...",tic AAB.TO ABX.TO ACD.TO AC...
1,2004-12-20,21.684490,29.090000,28.629999,28.629999,892100.0,ABX.TO,0.0,"[[0.009335785113573457, -0.0001770641735861586...",tic AAB.TO ABX.TO ACD.TO AC...
2,2004-12-20,4.109743,8.100000,8.100000,8.100000,1000.0,ACD.TO,0.0,"[[0.009335785113573457, -0.0001770641735861586...",tic AAB.TO ABX.TO ACD.TO AC...
3,2004-12-20,17.205807,24.150000,23.100000,23.799999,22143.0,ACX.TO,0.0,"[[0.009335785113573457, -0.0001770641735861586...",tic AAB.TO ABX.TO ACD.TO AC...
4,2004-12-20,12.918009,17.040001,16.670000,16.790001,217000.0,AEM.TO,0.0,"[[0.009335785113573457, -0.0001770641735861586...",tic AAB.TO ABX.TO ACD.TO AC...


In [78]:
train = data_split(df, '2004-01-01','2020-12-39')
#trade = data_split(df, '2020-01-01', config.END_DATE)

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [79]:
train.head()

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,date,close,high,low,open,volume,tic,day,cov_list,return_list
0,2004-12-20,0.415720,0.450000,0.450000,0.450000,0.0,AAB.TO,0.0,"[[0.009335785113573457, -0.0001770641735861586...",tic AAB.TO ABX.TO ACD.TO AC...
0,2004-12-20,21.684490,29.090000,28.629999,28.629999,892100.0,ABX.TO,0.0,"[[0.009335785113573457, -0.0001770641735861586...",tic AAB.TO ABX.TO ACD.TO AC...
0,2004-12-20,4.109743,8.100000,8.100000,8.100000,1000.0,ACD.TO,0.0,"[[0.009335785113573457, -0.0001770641735861586...",tic AAB.TO ABX.TO ACD.TO AC...
0,2004-12-20,17.205807,24.150000,23.100000,23.799999,22143.0,ACX.TO,0.0,"[[0.009335785113573457, -0.0001770641735861586...",tic AAB.TO ABX.TO ACD.TO AC...
0,2004-12-20,12.918009,17.040001,16.670000,16.790001,217000.0,AEM.TO,0.0,"[[0.009335785113573457, -0.0001770641735861586...",tic AAB.TO ABX.TO ACD.TO AC...


In [80]:
import numpy as np
import pandas as pd
from gym.utils import seeding
import gym
from gym import spaces
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from stable_baselines3.common.vec_env import DummyVecEnv


class StockPortfolioEnv(gym.Env):
    """A single stock trading environment for OpenAI gym

    Attributes
    ----------
        df: DataFrame
            input data
        stock_dim : int
            number of unique stocks
        hmax : int
            maximum number of shares to trade
        initial_amount : int
            start money
        transaction_cost_pct: float
            transaction cost percentage per trade
        reward_scaling: float
            scaling factor for reward, good for training
        state_space: int
            the dimension of input features
        action_space: int
            equals stock dimension
        tech_indicator_list: list
            a list of technical indicator names
        turbulence_threshold: int
            a threshold to control risk aversion
        day: int
            an increment number to control date

    Methods
    -------
    _sell_stock()
        perform sell action based on the sign of the action
    _buy_stock()
        perform buy action based on the sign of the action
    step()
        at each step the agent will return actions, then
        we will calculate the reward, and return the next observation.
    reset()
        reset the environment
    render()
        use render to return other functions
    save_asset_memory()
        return account value at each time step
    save_action_memory()
        return actions/positions at each time step


    """
    metadata = {'render.modes': ['human']}

    def __init__(self,
                df,
                stock_dim,
                hmax,
                initial_amount,
                transaction_cost_pct,
                reward_scaling,
                state_space,
                action_space,
                tech_indicator_list,
                turbulence_threshold=None,
                lookback=252,
                day = 0):
        #super(StockEnv, self).__init__()
        #money = 10 , scope = 1
        self.day = day
        self.lookback=lookback
        self.df = df
        self.stock_dim = stock_dim
        self.hmax = hmax
        self.initial_amount = initial_amount
        self.transaction_cost_pct =transaction_cost_pct
        self.reward_scaling = reward_scaling
        self.state_space = state_space
        self.action_space = action_space
        self.tech_indicator_list = tech_indicator_list

        # action_space normalization and shape is self.stock_dim
        self.action_space = spaces.Box(low = 0, high = 1,shape = (self.action_space,))
        # Shape = (34, 30)
        # covariance matrix + technical indicators
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape = (self.state_space+len(self.tech_indicator_list),self.state_space))

        # load data from a pandas dataframe
        self.data = self.df.loc[self.day,:]
        self.covs = self.data['cov_list'].values[0]
        self.state =  np.append(np.array(self.covs), [self.data[tech].values.tolist() for tech in self.tech_indicator_list ], axis=0)
        self.terminal = False
        self.turbulence_threshold = turbulence_threshold
        # initalize state: inital portfolio return + individual stock return + individual weights
        self.portfolio_value = self.initial_amount

        # memorize portfolio value each step
        self.asset_memory = [self.initial_amount]
        # memorize portfolio return each step
        self.portfolio_return_memory = [0]
        self.actions_memory=[[1/self.stock_dim]*self.stock_dim]
        self.date_memory=[self.data.date.unique()[0]]


    def step(self, actions):
        # print(self.day)
        self.terminal = self.day >= len(self.df.index.unique())-1
        # print(actions)

        if self.terminal:
            df = pd.DataFrame(self.portfolio_return_memory)
            df.columns = ['daily_return']
            plt.plot(df.daily_return.cumsum(),'r')
            plt.savefig('results/cumulative_reward.png')
            plt.close()

            plt.plot(self.portfolio_return_memory,'r')
            plt.savefig('results/rewards.png')
            plt.close()

            print("=================================")
            print("begin_total_asset:{}".format(self.asset_memory[0]))
            print("end_total_asset:{}".format(self.portfolio_value))

            df_daily_return = pd.DataFrame(self.portfolio_return_memory)
            df_daily_return.columns = ['daily_return']
            if df_daily_return['daily_return'].std() !=0:
              sharpe = (252**0.5)*df_daily_return['daily_return'].mean()/ \
                       df_daily_return['daily_return'].std()
              print("Sharpe: ",sharpe)
            print("=================================")

            return self.state, self.reward, self.terminal,{}

        else:
            #print("Model actions: ",actions)
            # actions are the portfolio weight
            # normalize to sum of 1
            #if (np.array(actions) - np.array(actions).min()).sum() != 0:
            #  norm_actions = (np.array(actions) - np.array(actions).min()) / (np.array(actions) - np.array(actions).min()).sum()
            #else:
            #  norm_actions = actions
            weights = self.softmax_normalization(actions)
            #print("Normalized actions: ", weights)
            self.actions_memory.append(weights)
            last_day_memory = self.data

            #load next state
            self.day += 1
            self.data = self.df.loc[self.day,:]
            self.covs = self.data['cov_list'].values[0]
            self.state =  np.append(np.array(self.covs), [self.data[tech].values.tolist() for tech in self.tech_indicator_list ], axis=0)
            #print(self.state)
            # calcualte portfolio return
            # individual stocks' return * weight
            portfolio_return = sum(((self.data.close.values / last_day_memory.close.values)-1)*weights)
            # update portfolio value
            new_portfolio_value = self.portfolio_value*(1+portfolio_return)
            self.portfolio_value = new_portfolio_value

            # save into memory
            self.portfolio_return_memory.append(portfolio_return)
            self.date_memory.append(self.data.date.unique()[0])
            self.asset_memory.append(new_portfolio_value)

            # the reward is the new portfolio value or end portfolo value
            self.reward = new_portfolio_value
            #print("Step reward: ", self.reward)
            #self.reward = self.reward*self.reward_scaling

        return self.state, self.reward, self.terminal, {}

    def reset(self):
        self.asset_memory = [self.initial_amount]
        self.day = 0
        self.data = self.df.loc[self.day,:]
        # load states
        self.covs = self.data['cov_list'].values[0]
        self.state =  np.append(np.array(self.covs), [self.data[tech].values.tolist() for tech in self.tech_indicator_list ], axis=0)
        self.portfolio_value = self.initial_amount
        #self.cost = 0
        #self.trades = 0
        self.terminal = False
        self.portfolio_return_memory = [0]
        self.actions_memory=[[1/self.stock_dim]*self.stock_dim]
        self.date_memory=[self.data.date.unique()[0]]
        return self.state

    def render(self, mode='human'):
        return self.state

    def softmax_normalization(self, actions):
        numerator = np.exp(actions)
        denominator = np.sum(np.exp(actions))
        softmax_output = numerator/denominator
        return softmax_output


    def save_asset_memory(self):
        date_list = self.date_memory
        portfolio_return = self.portfolio_return_memory
        #print(len(date_list))
        #print(len(asset_list))
        df_account_value = pd.DataFrame({'date':date_list,'daily_return':portfolio_return})
        return df_account_value

    def save_action_memory(self):
        # date and close price length must match actions length
        date_list = self.date_memory
        df_date = pd.DataFrame(date_list)
        df_date.columns = ['date']

        action_list = self.actions_memory
        df_actions = pd.DataFrame(action_list)
        df_actions.columns = self.data.tic.values
        df_actions.index = df_date.date
        #df_actions = pd.DataFrame({'date':date_list,'actions':action_list})
        return df_actions

    def _seed(self, seed=None):
        self.np_random, seed = seeding.np_random(seed)
        return [seed]

    def get_sb_env(self):
        e = DummyVecEnv([lambda: self])
        obs = e.reset()
        return e, obs

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [81]:
stock_dimension = len(train.tic.unique())
state_space = stock_dimension
print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

Stock Dimension: 133, State Space: 133


In [82]:
print(train.shape)
print(train.dtypes)
print(train.head())

(95153, 10)
date            object
close          float64
high           float64
low            float64
open           float64
volume         float64
tic             object
day            float64
cov_list        object
return_list     object
dtype: object
         date      close       high        low       open    volume     tic  \
0  2004-12-20   0.415720   0.450000   0.450000   0.450000       0.0  AAB.TO   
0  2004-12-20  21.684490  29.090000  28.629999  28.629999  892100.0  ABX.TO   
0  2004-12-20   4.109743   8.100000   8.100000   8.100000    1000.0  ACD.TO   
0  2004-12-20  17.205807  24.150000  23.100000  23.799999   22143.0  ACX.TO   
0  2004-12-20  12.918009  17.040001  16.670000  16.790001  217000.0  AEM.TO   

   day                                           cov_list  \
0  0.0  [[0.009335785113573457, -0.0001770641735861586...   
0  0.0  [[0.009335785113573457, -0.0001770641735861586...   
0  0.0  [[0.009335785113573457, -0.0001770641735861586...   
0  0.0  [[0.0093357851135

In [83]:
print(train.iloc[0]['cov_list'])
print(train.iloc[0]['return_list'])


[[ 9.33578511e-03 -1.77064174e-04 -2.41722955e-04 ...  1.09106765e-04
  -9.57817708e-05 -4.40586243e-04]
 [-1.77064174e-04  2.18624177e-04  9.41348069e-07 ... -1.08353227e-05
   7.11334593e-05  7.16566591e-05]
 [-2.41722955e-04  9.41348069e-07  7.88279562e-04 ... -7.05087166e-06
   4.26553707e-05 -1.78650959e-04]
 ...
 [ 1.09106765e-04 -1.08353227e-05 -7.05087166e-06 ...  2.21115191e-04
  -1.91547895e-04  2.05752034e-06]
 [-9.57817708e-05  7.11334593e-05  4.26553707e-05 ... -1.91547895e-04
   3.95109288e-03  1.99793837e-04]
 [-4.40586243e-04  7.16566591e-05 -1.78650959e-04 ...  2.05752034e-06
   1.99793837e-04  3.16168905e-03]]
tic           AAB.TO    ABX.TO    ACD.TO    ACX.TO    AEM.TO    AFN.TO  \
date                                                                     
2004-07-01  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000   
2004-07-02  0.000000 -0.003763  0.001126  0.003861  0.010693  0.004897   
2004-07-05  0.000000 -0.001888  0.000000 -0.007693 -0.011136  0.0009

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [84]:
import numpy as np

# Extract the diagonal of the covariance matrix
train['cov_diag'] = train['cov_list'].apply(lambda x: np.diag(x) if isinstance(x, (list, np.ndarray)) else np.nan)

# Expand the diagonal values into separate columns
cov_df = train['cov_diag'].apply(pd.Series)
cov_df.columns = [f'cov_var_{i}' for i in range(cov_df.shape[1])]

# Merge back and drop old columns
train = pd.concat([train.drop(['cov_list', 'cov_diag'], axis=1), cov_df], axis=1)

print(train.head())


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


         date      close       high        low       open    volume     tic  \
0  2004-12-20   0.415720   0.450000   0.450000   0.450000       0.0  AAB.TO   
0  2004-12-20  21.684490  29.090000  28.629999  28.629999  892100.0  ABX.TO   
0  2004-12-20   4.109743   8.100000   8.100000   8.100000    1000.0  ACD.TO   
0  2004-12-20  17.205807  24.150000  23.100000  23.799999   22143.0  ACX.TO   
0  2004-12-20  12.918009  17.040001  16.670000  16.790001  217000.0  AEM.TO   

   day                                        return_list  cov_var_0  ...  \
0  0.0  tic           AAB.TO    ABX.TO    ACD.TO    AC...   0.009336  ...   
0  0.0  tic           AAB.TO    ABX.TO    ACD.TO    AC...   0.009336  ...   
0  0.0  tic           AAB.TO    ABX.TO    ACD.TO    AC...   0.009336  ...   
0  0.0  tic           AAB.TO    ABX.TO    ACD.TO    AC...   0.009336  ...   
0  0.0  tic           AAB.TO    ABX.TO    ACD.TO    AC...   0.009336  ...   

   cov_var_122  cov_var_123  cov_var_124  cov_var_125  cov_var

In [85]:
print(train.columns)


Index(['date', 'close', 'high', 'low', 'open', 'volume', 'tic', 'day',
       'return_list', 'cov_var_0',
       ...
       'cov_var_122', 'cov_var_123', 'cov_var_124', 'cov_var_125',
       'cov_var_126', 'cov_var_127', 'cov_var_128', 'cov_var_129',
       'cov_var_130', 'cov_var_131'],
      dtype='object', length=141)


/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [86]:
if 'cov_list' in train.columns:
    train = train.drop(columns=['cov_list'])

In [89]:
state_space = len(train.columns) - 2  # Excluding 'date' and 'tic'
env_kwargs["state_space"] = state_space

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [90]:
env_kwargs = {
    "hmax": 100,
    "initial_amount": 1000000,
    "transaction_cost_pct": 0.001,
    "state_space": state_space,
    "stock_dim": stock_dimension,
    "tech_indicator_list": config.INDICATORS,
    "action_space": stock_dimension,
    "reward_scaling": 1e-4

}

e_train_gym = StockPortfolioEnv(df = train, **env_kwargs)

/usr/local/lib/python3.11/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


KeyError: 'cov_list'

In [ ]:
import pandas as pd

# Expand covariance list into separate columns
cov_df = train['cov_list'].apply(pd.Series)
cov_df.columns = [f'cov_{i}' for i in range(cov_df.shape[1])]

# Expand return list into separate columns
return_df = train['return_list'].apply(pd.Series)
return_df.columns = [f'return_{i}' for i in range(return_df.shape[1])]

# Merge back into main dataset and drop original columns
train = pd.concat([train.drop(['cov_list', 'return_list'], axis=1), cov_df, return_df], axis=1)

print(train.head())  # Check if the structure looks correct now
